In [ ]:
### Ham / Spam 분류 모델 파인 튜닝
  - SpamDataset 클래스
  - util : create_balanced_dataset(), random_split(), calc_accuracy_loader()
  - train_classifier_simple()
  - 모델 로딩 / 파인튜팅 / 학습 / 평가 / 시각화

In [ ]:
### 유틸리티 함수

## 데이터 개수를 균형있게 만드는 함수 (Ham/Spam)
def create_balanced_dataset( df ):
    # spam 라벨 개수 계산
    num_spam = df[ df["Label"] == "spam" ].shape[0]     # row

    # ham 데이터 중 spam 개수만큼 램던 추출
    ham_dataset = df[ df["Label"] == "ham" ].sample( num_spam, random_state=123 )

    # 데이터 병합
    balanced_df = pd.concat( [ ham_dataset, df[ df["Label"] == "spam"] ])

    return balanced_df


## 데이터 셋을 train / val / test 으로 나누는 함수
def random_split( df, train_frac, val_frac ) :
    # 전체 데이터 섞기
    df = df.sample( frac=1, random_state=123 ).reset_index( drop=True )

    # 분할 지점 인덱스 계산
    train_end = int( len(df) * train_frac )
    val_end = train_end + int( len(df) * val_frac )

    # 데이터 분할
    train_df = df[ : train_end ]
    val_df = df[ train_end : val_end ]
    test_df = df[ val_end : ]

    return train_df, val_df, test_df


## 데이터 셋 전체에 대해 정확도 계산
def calc_accuracy_loader( data_loader, model, device, num_batches=None ) :
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))

    # 배치 사이즈만큼 루프 돌면서 평가 결과로 정확도 계산
    for i, (input_batch, target_batch) in enumerate(data_loader) :
        if i < num_batches:
            with torch.no_grad():
                logits = model( input_batch )
                logits = logits[:, -1, :]

            predicted_labels = torch.argmax( logits, dim=-1 )

            num_examples += predicted_labels.shape[ 0 ]     # B
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break

    return correct_predictions / num_examples     # 정확도 accuracy = # of True / # of Item

In [ ]:
### SpamDataset

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        ## 1. 텍스트 -> 토큰 id 리스트로 변환
        self.encoded_texts = [ tokenizer.encode(text) for text in self.data["Text"] ]

        ## 2. 최대 길이 설정
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            max_length = max_length

            # max length 보다 긴 문장 클리핑
            self.encoded_texts = [ encoded_text[ : self.max_length ] for encoded_text in self.encoded_texts ]

        ## 3. 모든 sequence 길이를 max length 로 맞추고 패딩 추가
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __len__(self):
        return len(self.encoded_texts)

    def __getitem__(self, index):
        encoded = self.encoded_texts[ index ]
        label = self.data.iloc[ index ][ "Label" ]      # 0, 1

        # 텐서로 변환해서 리턴
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tenser(label, dtype=torch.long)
        )

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length >= max_length:
                max_length = encoded_length
        
        return max_length

In [ ]:
### 분로 모델 학습 함수

def train_classifier_simple( model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter ) :
    # 기록용 변수들
    train_losses, val_losses, train_accs, val_accs = [], [], [], []
    example_seen, global_step = 0, -1

    # 에폭 단위로 학습
    for epoch in range(num_epochs):
        model.train()   # 학습 모드로 설정

        # 배치 단위로 학습
        for input_batch, target_batch in train_loader:
            optimizer.zero_grad()   # 기울기 초기화
            loss = calc_loss_loader( input_batch, target_batch, model, device )     # input_batch : [B, N], target_batch : [ N ] (0 or 1) => flatten 필요없음
            loss.backward()         # 역전파 (기울기 계산)
            optimizer.step()        # 가중치 업데이트
            
            example_seen += input_batch.shape[0]
            global_step += 1

            ## 주기적으로 평가
            if global_step / eval_freq == 0 :
                train_loss, val_loss = evaluate_model(...)
                train_losses.append( train_loss )
                val_losses.append( val_loss )

        # 1 에폭마다 정확도 계산
        train_accuracy = calc_accuracy_loader( train_loader, model, device, num_batches=eval_iter )
        val_accuracy = calc_accuracy_loader( val_loader, model, device, num_batches=eval_iter )
        train_accs.append( train_accuracy )
        val_accs.append( val_accuracy )

    return train_losses, val_losses, train_accs, val_accs

In [ ]:
### 모델 로딩 및 파인 튜닝해서 학습 후 평가 / 시각화

## 1. 데이터 다운로드 및 준비
download_and_unzip_spam_data(url, ...)

# 1.1 데이터 로드 및 전처리
df = pd.read_csv( data_file, sep="\t", header=None, names=["Label", "Text"])
balanced_df = create_balanced_dataset( df )
balanced_df["Label"] = balanced_df["Label"].map({ "ham": 0, "spam": 1 })    # ham/spam -> 0, 1

# 1.2 train / val / test 데이터셋 분리 및 csv 저장
train_df, val_df, test_df = random_split( balanced_df, 0.7, 0.1 )
train_df.to_csv("datas/train.csv", index=None)
val_df.to_csv("datas/val.csv", index=None)
test_df.to_csv("datas/test.csv", index=None)


## 2. 데이터 셋 / 데이터 로더 생성
tokenizer = tiktoken.get_encoding("gpt2")

# 2.1 데이터 셋 생성
train_dataset = SpamDataset( csv_file="datas/train.csv", max_length=None, tokenizer=tokenizer)  # max_length 이 가장 긴 토큰으로 설정됨
val_dataset = SpamDataset( csv_file="datas/val.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)  # max_length 를 train_dataset 에 맞춤
test_dataset = SpamDataset( csv_file="datas/test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)  # max_length 를 train_dataset 에 맞춤

num_workers = 0
batch_size = 8
torch.manual_seed(123)

# 2.2 데이터 로더 생성
train_loader = DataLoader( train_dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=num_workers )    # shuffle/drop_last True
val_loader = DataLoader( val_dataset, batch_size=batch_size, shuffle=False, drop_last=False )    # shuffle/drop_last False
test_loader = DataLoader( test_dataset, batch_size=batch_size, shuffle=False, drop_last=False )    # shuffle/drop_last False


## 3. 사전 학습된 모델 로딩
CHOOSE_MODEL = "gpt2-small (124M)"

BASE_CONFIG = {
    "vocab_size": 50257,     
    "context_length": 1024,  
    "drop_rate": 0.0,        
    "qkv_bias": True         
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    # 다른 모델 사이즈 설정들...
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

# 데이터 셋의 길이가 모델의 context length 초과하는지 체크
assert train_dataset.max_length <= BASE_CONFIG["context_length"], ("error")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "gpt2-small-124M.pth"
model = GPTModel(BASE_CONFIG)
checkpoint = torch.load(model_name, map_location=device)
model.load_state_dict(checkpoint)
model.to(device)


## 4. 모델 수정 및 파인 튜닝해서

torch.manual_seed(123)

# 4.1 모든 파라미터를 고정하여 학습 하지 않도록 설정
for param in model.parameters() :
    param.requires_grad = False

# 4.2 출력 층 교체 50257 -> 2 개 예측
num_classes = 2     # ham / spam
model.out_head = torch.nn.Linear( in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

# 4.3 마지막 transformer 와 final norm 을 학습 가능하도록 설정
for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

model.to(device)


## 5. 파인 튜닝된 모델 학습
start_time = time.time()

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)

num_epochs = 5

train_losses, val_losses, train_accs, val_accs = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device, num_epochs=num_epochs, eval_freq=50, eval_iter=5
)

end_time = time.time()


## 6. 결과 시각화
# 6.1 loss graph
epochs_tenser = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tenser = torch.linspace(0, examples_seen, len(train_losses))
plot_values(epochs_tenser, examples_seen_tenser, train_losses, val_losses)